In [1]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import re
from openpyxl import load_workbook

# ===============================
# 1) Daten einlesen
# ===============================
# Eingabedatei mit bibliografischen Daten (z. B. IDN, Titel, Schlagworte)
input_file = "Zwischenauswertung1.xlsx"
# Mapping-Datei mit Schlagworten und Gewichtungen
mapping_file = "schlagworte_ddc_gewichtung.xlsx"
# Ausgabe-Datei für die finalen Ergebnisse
output_file = "ergebnis_juni_2026.xlsx"
# Hauptdaten werden als DataFrame eingelesen (alle Spalten als String)
idn_df = pd.read_excel(input_file, dtype=str)
# Mapping-Tabelle laden
mapping_df = pd.read_excel(mapping_file, dtype=str)

# Liste aller relevanten Schlagwörter (nur Typ = "Schlagwort")
keywords = mapping_df[mapping_df["Typ"] == "Schlagwort"]["Begriff"].dropna().tolist()
# Gewichtung je Begriff (wird später für Scoring genutzt)
weights = mapping_df.set_index("Begriff")["Gewichtung"].astype(int).to_dict()
# Zuordnung Begriff → Systematik (z. B. Klassifikation )
systematics = mapping_df.set_index("Begriff")["Systematik"].to_dict()



# ===============================
# 2) Inhaltsverzeichnisse durchsuchen
# ===============================
# Diese Funktion durchsucht die DNB-Text- und PDF-Ansichten nach Schlagwörtern
# DNB-URLs für Text- und PDF-Ansicht eines Titels werden erstellt und genutzt, Ergebnisse der gefunden Schlagwörter und die Gesamtgewichtung ausgegeben
def find_keywords(idn):
    url_text = f"https://d-nb.info/{idn}/04/text"
    url_pdf = f"https://d-nb.info/{idn}/04/pdf"
    formatted = ""
    total_weight = 0
    best_syst = None
    #HTTP-Request auf DNB-Textseite
    try:
        response = requests.get(url_text, timeout=10)
        if response.status_code == 200:
            soup = BeautifulSoup(response.text, "html.parser")
            text = soup.get_text().lower()
            # Dictionary für gefundene Wörter + Metadaten
            word_counts = {}
            for word in keywords:
                pattern = re.compile(rf"\b\w*{re.escape(word.lower())}\w*\b", re.IGNORECASE)
                matches = pattern.findall(text)
                count = len(matches)
                if count > 0:
                    word_counts[word] = (count, weights.get(word, 1), systematics.get(word, ""))

            if word_counts:
                total_weight = sum(v[1] for v in word_counts.values())
                best_word, (best_count, best_weight, best_syst) = max(
                    word_counts.items(),
                    key=lambda x: (x[1][1], x[1][0])
                )
                formatted = "; ".join(
                    f"{w} (Gewicht={wt}, Treffer={ct})"
                    for w, (ct, wt, _) in word_counts.items()
                )
    # Fehlerbehandlung bei Netzwerkproblemen
    except requests.RequestException:
        formatted = "Fehler"
    # Rückgabe als neue Spalte in der Exceltabelle
    # - Text-URL
    # - PDF-URL
    # - gefundene Schlagwörter
    # - Gesamtgewicht
    # - beste Systematik
    return url_text, url_pdf, formatted, total_weight, best_syst or ""

idn_df[["URL", "PDF-URL", "Gefundene Schlagwörter", "Gesamtgewichtung", "Gewinner-Systematik"]] = \
    idn_df["IDN"].apply(find_keywords).apply(pd.Series)

# ===============================
# 3) Optional : Schlagwort-Mapping auf Titel anwenden
# ===============================
# Diese Funktion ergänzt zusätzlich Schlagwörter im Titel zur Gesamtgewichtung. Falls dies Ausgangsdatei keine Titelspalte aufweist muss dies entfallen. 
def apply_mapping(row):
    total_weight = row["Gesamtgewichtung"]
    systematik = row["Gewinner-Systematik"]
    found_extra = []

    # --- Titel durchsuchen ---
    if pd.notna(row["Titel"]):
        t = row["Titel"].lower()
        for w in keywords:
            if re.search(rf"\b{re.escape(w.lower())}\b", t):
                total_weight += weights.get(w, 1)
                found_extra.append(f"Titel: {w}")
                if not systematik:
                    systematik = systematics.get(w)


    # --- Schlagwort-Spalte durchsuchen (exakt) ---
    if pd.notna(row["Schlagwort"]):
        s_list = [s.strip() for s in row["Schlagwort"].split(",")]
        for s in s_list:
            if s in systematics:
                systematik = systematics[s]
                found_extra.append(f"Schlagwort: {s}")

    return pd.Series([total_weight, systematik or "", "; ".join(found_extra)])

# --- Anwendung auf DataFrame ---
idn_df[["Gesamtgewichtung", "Gewinner-Systematik", "Zusätzliche Treffer"]] = \
    idn_df.apply(apply_mapping, axis=1)
# ===============================
# 4) Filtern nach Gesamtgewichtung ≥ 3
# ===============================
# Nur Datensätze behalten, die mindestens Gewicht 3 erreichen, dieser Wert ist hier einfach einstellbar, aufgrund eines weiteren Skriptes welches die Ergebnis mithilfe von KI eingrenzt wurde hier ein großzügiger wert von 3 genutzt.
idn_df = idn_df[idn_df["Gesamtgewichtung"] >= 3]

# ===============================
# 5) Excel speichern + Hyperlinks setzen
# ===============================
# Ergebnis als Excel-Datei speichern
idn_df.to_excel(output_file, index=False)

wb = load_workbook(output_file)
ws = wb.active
col_url = [c[0] for c in enumerate(ws[1]) if c[1].value == "URL"][0] + 1
col_pdf = [c[0] for c in enumerate(ws[1]) if c[1].value == "PDF-URL"][0] + 1

for row in range(2, ws.max_row + 1):
    if ws.cell(row=row, column=col_url).value:
        ws.cell(row=row, column=col_url).hyperlink = ws.cell(row=row, column=col_url).value
        ws.cell(row=row, column=col_url).style = "Hyperlink"
    if ws.cell(row=row, column=col_pdf).value:
        ws.cell(row=row, column=col_pdf).hyperlink = ws.cell(row=row, column=col_pdf).value
        ws.cell(row=row, column=col_pdf).style = "Hyperlink"

wb.save(output_file)
print(f"Fertig! Ergebnisse gespeichert in {output_file}")

# ===============================
# 6) Optional: Ergebnis-Excel aufteilen
# ===============================
# für die weitere Verarbeitung mithilfe einer KI, benötigen wir kleine Blöcke um innerhalb des täglichen Promptlimits zu arbeiten
import os
import pandas as pd

chunk_size = 100  # Treffer pro Datei

df = pd.read_excel(output_file, dtype=str)

total_rows = len(df)
if total_rows == 0:
    raise ValueError("Die Excel-Datei enthält keine Daten.")

base_name, ext = os.path.splitext(output_file)

file_count = 0

for i in range(0, total_rows, chunk_size):
    file_count += 1
    chunk_df = df.iloc[i:i + chunk_size]

    output_chunk = f"{base_name}_{file_count}{ext}"
    chunk_df.to_excel(output_chunk, index=False)

    print(f"Erstellt: {output_chunk} ({len(chunk_df)} Treffer)")

print(f"\nFertig! {file_count} Dateien erzeugt.")


C:\Users\sickerti\AppData\Local\Temp\ipykernel_16764\744768193.py:46: MarkupResemblesLocatorWarning: The input looks more like a filename than markup. You may want to open this file and pass the filehandle into Beautiful Soup.
  soup = BeautifulSoup(response.text, "html.parser")


Fertig! Ergebnisse gespeichert in ergebnis_juni_2026.xlsx
Erstellt: ergebnis_juni_2026_1.xlsx (100 Treffer)
Erstellt: ergebnis_juni_2026_2.xlsx (75 Treffer)

Fertig! 2 Dateien erzeugt.
